Process sessions
Jira: https://keyless.atlassian.net/browse/BIOM-625

In [1]:
import shutil

import numpy as np
import pandas as pd
from loguru import logger

from distutils.dir_util import copy_tree
from pathlib import Path

import cv2
import datasets
import os
import s3fs

import modules.globals
from modules import core
from modules.face_analyser import get_one_face

from tqdm.auto import tqdm

2025-07-12 06:27:57.863065: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-12 06:27:57.863112: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-12 06:27:57.863139: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
ds_root = "s3://sagemaker-production-eu-central-1-kl-biometric-datasets/raw_datasets/face_biometrics/deepfakes/hackathon_2025-07_deepfakes/to_process_to_create_dest_dataset/"
ds_key = "train_aggregation_ensemble/part_0"
output_ds_root = "s3://sagemaker-production-eu-central-1-kl-biometric-datasets/raw_datasets/face_biometrics/deepfakes/hackathon_2025-07_deepfakes/destination_datasets/"

local_original_imgs_root = "/home/sagemaker-user/face_swap/original_images/"
local_swapped_imgs_root = "/home/sagemaker-user/face_swap/swapped_images_enh/"
execution_provider = "cpu"  # cuda or cpu
face_enhancer = False
rng_seed = 42

In [3]:
### init ###
modules.globals.execution_providers = core.decode_execution_providers(
    [execution_provider]
)
frame_processors = ["face_swapper"]
modules.globals.fp_ui["face_enhancer"] = False
if face_enhancer:
    frame_processors.append("face_enhancer")
    modules.globals.fp_ui["face_enhancer"] = True
np.random.seed(rng_seed)

modules.globals.max_memory = core.suggest_max_memory()
modules.globals.execution_threads = core.suggest_execution_threads()
core.limit_resources()

In [4]:
ds_path = os.path.join(ds_root, ds_key, "hf_dataset")
ds = datasets.Dataset.load_from_disk(ds_path)

/home/sagemaker-user/Deep-Live-Cam/.venv/lib/python3.10/site-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


In [5]:
local_swapped_imgs_path = os.path.join(local_swapped_imgs_root, ds_key)
os.makedirs(local_swapped_imgs_path, exist_ok=True)


In [7]:
session_img_paths = {}
for row in tqdm(ds):
    session_folder = row["session_folder"]
    session_local_path = os.path.join(local_swapped_imgs_path, session_folder)

    if session_folder not in session_img_paths:
        session_img_paths[session_folder] = []
        os.makedirs(session_local_path, exist_ok=True)
        source_img_path = os.path.join(
            session_local_path, "source_img.jpg"
        )
        row["source_img_raw"].save(source_img_path)

    img_path = os.path.join(session_local_path, row["photo_name"])
    if not img_path.endswith == ".jpg":
        img_path += ".jpg"
    row["img_raw"].save(img_path)

    session_img_paths[session_folder].append(img_path)

  0%|          | 0/7454 [00:00<?, ?it/s]

In [8]:
for session_folder in session_img_paths.keys():
    logger.info(f"Processing session {session_folder}")
    source_img_path = os.path.join(
        os.path.dirname(session_img_paths[session_folder][0]), "source_img.jpg"
    )

    for frame_processor in core.get_frame_processors_modules(frame_processors):
        logger.info(f"Progressing... {frame_processor.NAME}")
        print(f"Total frames: {len(session_img_paths[session_folder])}")
        frame_processor.process_video(
            source_img_path, session_img_paths[session_folder]
        )
        core.release_resources()

2025-07-12 06:32:33.465 | INFO     | __main__:<module>:2 - Processing session T-00056_1641671148937_1.1.1
2025-07-12 06:32:33.467 | INFO     | __main__:<module>:8 - Progressing... DLC.FACE-SWAPPER


Total frames: 50


Processing:   0%|          | 0/50 [00:00<?, ?frame/s, execution_providers=['CPUExecutionProvider'], execution_threads=8, max_memory=16]

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sagemaker-user/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sagemaker-user/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/sagemaker-user/.insightface/models/buf

Processing: 100%|██████████| 50/50 [01:33<00:00,  1.86s/frame, execution_providers=['CPUExecutionProvider'], execution_threads=8, max_memory=16]
2025-07-12 06:34:06.628 | INFO     | __main__:<module>:2 - Processing session T-00056_1645359712782_1.1.1
2025-07-12 06:34:06.628 | INFO     | __main__:<module>:8 - Progressing... DLC.FACE-SWAPPER


Total frames: 50


Processing:  66%|██████▌   | 33/50 [00:56<00:23,  1.38s/frame, execution_providers=['CPUExecutionProvider'], execution_threads=8, max_memory=16]